
# Imputation Redux: Imputing Missing Categorical Data

You might have realized that because `OneHotEncoder` fails with missing values, and Scikit-Learn prefers to work with numerical data, we face a bit of a conundrum when trying to deal with missing categorical values. Note that we really don't want to use these imputers after a one-hot encoding either, because there is is no guarantee these imputers will follow the implicit rule that only one column of a one-hot encoded categorical set can be `1`. Here are two strategies.

## Using Pandas or Most Frequent Category

You can use Pandas to replace nulls, using one of the methods we covered previously. Alternatively, you can use `SimpleImputer` with the `strategy='most_frequent'` option to impute missing values with the most frequent category in each column before one-hot encoding.


In [1]:
import numpy, pandas, sklearn, scipy
print(numpy.__version__)
print(pandas.__version__)
print(sklearn.__version__)
print(scipy.__version__)
#1.26.4
#2.2.2
#1.5.1
#1.13.1

2.4.4
3.0.2
1.8.0
1.17.1


In [6]:
#pip install --upgrade scikit-learn scipy pandas numpy

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 8.0/8.0 MB 49.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ---------------- ----------------------- 14.7/36.5 MB 71.0 MB/s eta 0:00:01
   -------------------------------- ------- 29.9/36.5 MB 73.0 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.5 MB 72.4 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 50.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 60.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------  12.1/12.3 MB 75.1 MB/s eta 0:00:01
   ---------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.4 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.4.4 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.17.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.2 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.32.0 which is incompatible.


In [2]:
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

# Create DataFrame with missing values
df = pd.DataFrame({
    'color': ['red', 'blue', np.nan, 'red', np.nan, 'green'],
    'target': [1, 0, 1, 0, 0, 1]
})

# Impute missing values
imp = SimpleImputer(strategy='most_frequent')
df['color'] = imp.fit_transform(df[['color']])[:,0]
df

,color,target
0,red,1
1,blue,0
2,red,1
3,red,0
4,red,0
5,green,1


## Use a Different Library!

As you might imagine, others have struggled with this, and so there are other libraries designed to address this problem. For instance, the `fancyimpute` package has both a `KNNImputer` and an `IterativeImputer` you might try. Here's an example with the `KNNImputer` from `fancyimpute`.

This code installs the `fancyinpute` library since Google Colab does not already have it installed, which had been the case for all libraries we've worked with so far.

In [3]:
#!pip install fancyimpute
#print('installed')

Defaulting to user installation because normal site-packages is not writeable
installed



### ~~FancyImputes~~ K-Nearest Neighbors (KNN) Imputer

`KNN` ~~from `fancyimputer`~~ won't work with categorical data directly, but instead of using the `mean` (which is used by SciKit-Learn's `KNNImputer`) it uses the `mode` for imputation, which is what we want. To use `KNN`, first you should encode your data using an `OrdinalEncoder` or `LabelEncoder`, then impute, then transform your data back into the categorical values you want.  This is more complicated than it should be because there is no easy way to preserve nulls in your data.



In [8]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

data = {
    'Fruit': ['Apple', 'Banana', 'Cherry', 'Apple', None, 'Banana'],
    'Color': ['Red', 'Yellow', 'Red', None, 'Green', 'Yellow']
}
df = pd.DataFrame(data)

encoders = {}
df_encoded = df.copy()

for col in df.columns:
    le = LabelEncoder()
    not_null_mask = df[col].notnull()

    encoded_vals = le.fit_transform(df.loc[not_null_mask, col].astype(str))

    df_encoded[col] = np.nan
    df_encoded[col] = df_encoded[col].astype(float)
    df_encoded.loc[not_null_mask, col] = encoded_vals.astype(float)

    encoders[col] = le

# Use sklearn's KNNImputer instead of fancyimpute
knn_imputer = KNNImputer(n_neighbors=5)
df_imputed = knn_imputer.fit_transform(df_encoded)

# Round and decode
df_imputed = pd.DataFrame(np.round(df_imputed), columns=df.columns).astype(int)

for col in df.columns:
    df_imputed[col] = encoders[col].inverse_transform(df_imputed[col])

print(df_imputed)

    Fruit   Color
0   Apple     Red
1  Banana  Yellow
2  Cherry     Red
3   Apple  Yellow
4  Banana   Green
5  Banana  Yellow


Other strategies may be applied in a similar manner, after which you can one-hot encode your data, and proceed with additional processing!

Note that there is currently no elegant solution for imputation of categorical variables, and so if you want something more sophisticated than a `SimpleImputer` with a "most_frequent" strategy, you'll probably need to write some code. However, we can turn the above method into our own "Imputer" class like this:

In [9]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder
from fancyimpute import KNN
import pandas as pd
import numpy as np

class CategoricalKNNImputer(BaseEstimator, TransformerMixin):
    def __init__(self, include_numeric=False, include_cols = []):
        self.encoders = {}
        self.knn_imputer = KNN()
        self.include_numeric = include_numeric
        self.include_cols = include_cols

    def fit(self, X, y=None):
        X = X.copy()

        if self.include_numeric:
            self.cols = X.columns.tolist()
        else:
            self.cols = X.select_dtypes(include=['object', 'category']).columns.tolist()+self.include_cols

        for col in self.cols:
            le = LabelEncoder()
            not_null_mask = X[col].notnull()
            if not_null_mask.sum() > 0:  # Only if there are non-null values to fit
                X.loc[not_null_mask, col] = le.fit_transform(X.loc[not_null_mask, col].astype(str))
                self.encoders[col] = le
        return self

    def transform(self, X):
        X_original = X.copy()
        X = X.copy()

        for col in self.cols:
            if col in self.encoders:  # Only if encoder exists
                not_null_mask = X[col].notnull()
                X.loc[not_null_mask, col] = self.encoders[col].transform(X.loc[not_null_mask, col].astype(str))

        X_imputed = self.knn_imputer.fit_transform(X)
        X_imputed = pd.DataFrame(X_imputed, columns=X.columns)

        for col in self.cols:
            if col in self.encoders:  # Only if encoder exists
                X_imputed.loc[:, col] = np.round(X_imputed.loc[:, col])  # Rounding only categorical columns
                X_imputed[col] = X_imputed[col].astype(int)  # Converting to int before decoding
                X_imputed[col] = self.encoders[col].inverse_transform(X_imputed[col])

        if not self.include_numeric:
            replacements = [x for x in X.columns if x not in self.cols]
            #numeric_cols = X_original.select_dtypes(include=[np.number]).columns
            X_imputed[replacements] = X_original[replacements]

        return X_imputed

The details of the Python might be more than you can understand at this point, but you should be able to recognize roughly what's going on here; we're simply building a component that works with Scikit-Learn to do KNN based imputation on categorical columns. You can apply this just like other Scikit-Learn components, using `fit` and `transform`.

## Important Considerations

When training machine learning models with imputed data, it's crucial to follow best practices to ensure the robustness and generalizability of your models. Here’s a guide that covers considerations like data leakage, when to use imputed data, and other relevant aspects:

### 1. Data Splitting
Always split your dataset into training, validation (optional), and test sets before any imputation to avoid data leakage. Leakage occurs when information from the validation/test sets is used to inform any part of the modeling process, leading to overly optimistic performance estimates.

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

### 2. Imputation
Perform imputation separately on each set:
   - Fit the imputer on the training set.
   - Transform both the training and test sets.

```python
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
imputer.fit(X_train)  # Fit only on the training set

X_train_imputed = imputer.transform(X_train)  # Transform the training set
X_test_imputed = imputer.transform(X_test)  # Transform the test set
```

### 3. When to Use Imputed Data
- When the amount of missing data is not substantial, imputation can leverage the available information, which would otherwise be discarded if only complete cases are used.
- When the data are missing at random or missing completely at random, imputation can yield unbiased estimators.

### 4. When to Avoid Imputed Data
- When missingness is related to the unobserved value itself (missing not at random), imputation might introduce bias.
- When there are very few observed cases, imputation might overfit the training data, and it's better to use complete cases if available.

### 5. Model Evaluation
- Evaluate model performance on the test set with imputed values, focusing on metrics relevant to your specific problem.
- Consider performing sensitivity analyses by using different imputation methods and comparing the results.
- Additionally, assess the model's performance on only complete cases in the test set, to understand how much information is gained (or lost) due to imputation.

### 6. Other Considerations
- **Hyperparameter Tuning and Model Selection:** Conduct model selection and hyperparameter tuning using only the training set. Use techniques like cross-validation to assess model generalization on the training set before final evaluation on the test set.
- **Complex Imputation Methods:** More advanced imputation methods like model-based imputation or multiple imputations may provide better results but come with their assumptions and computational cost.
- **Documentation:** Document all the steps involved in the imputation process, the reasons for choosing a particular imputation method, and any assumptions made.
